# Test des cles API Gemini et Groq

Ce notebook teste separement tes cles API **Google Gemini** et **Groq** avec une requete simple pour chacune.

Etapes :
1. Installer les SDK
2. Configurer les cles API
3. Tester Gemini
4. Tester Groq

In [ ]:
!pip install -q google-genai groq

## Configuration des cles API

- Cle Gemini : obtenue sur https://aistudio.google.com/apikey
- Cle Groq : obtenue sur https://console.groq.com/keys

Astuce Colab : tu peux aussi stocker ces cles dans les *Secrets* de Colab (icone cle dans la barre laterale) sous les noms `GEMINI_API_KEY` et `GROQ_API_KEY`.

In [ ]:
import os
from getpass import getpass

def get_key(env_name, prompt):
    key = None
    try:
        from google.colab import userdata
        key = userdata.get(env_name)
    except Exception:
        pass
    if not key:
        key = getpass(prompt)
    os.environ[env_name] = key
    return key

gemini_key = get_key('GEMINI_API_KEY', 'Entre ta cle API Gemini: ')
groq_key = get_key('GROQ_API_KEY', 'Entre ta cle API Groq: ')

## Test de la cle Gemini

Utilise le SDK unifie `google-genai` (remplace l'ancien `google-generativeai`).

In [ ]:
from google import genai

gemini_client = genai.Client(api_key=gemini_key)

gemini_response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents="Reponds en une phrase : combien font 12 * 8 ?",
)
print('--- Reponse Gemini ---')
print(gemini_response.text)

## Test de la cle Groq

Groq propose une inference tres rapide. On utilise `llama-3.1-8b-instant`, generalement autorise par defaut sur toutes les cles API. Si un modele te renvoie une erreur "blocked at the project level", utilise la cellule suivante pour lister les modeles reellement autorises par ta cle.

In [ ]:
# Diagnostic : liste les modeles reellement autorises par ta cle API Groq
from groq import Groq

groq_client = Groq(api_key=groq_key)

for m in groq_client.models.list().data:
    print(m.id)

In [ ]:
from groq import Groq

groq_client = Groq(api_key=groq_key)

groq_response = groq_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[
        {'role': 'user', 'content': "Reponds en une phrase : combien font 12 * 8 ?"}
    ],
)
print('--- Reponse Groq ---')
print(groq_response.choices[0].message.content)

## Comparaison des trois modeles sur la meme question

On envoie exactement la meme question a **Claude**, **Gemini** et **Groq**, et on compare les reponses ainsi que le temps de reponse de chacun.

In [ ]:
!pip install -q anthropic

In [ ]:
anthropic_key = get_key('ANTHROPIC_API_KEY', 'Entre ta cle API Anthropic: ')

In [ ]:
import time
import anthropic

question = "Combien font 12 * 8 ? Reponds en une seule phrase."

results = []

# --- Claude ---
anthropic_client = anthropic.Anthropic(api_key=anthropic_key)
start = time.time()
claude_response = anthropic_client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=256,
    messages=[{'role': 'user', 'content': question}],
)
elapsed = time.time() - start
claude_text = ''.join(b.text for b in claude_response.content if b.type == 'text')
results.append(('Claude (haiku-4-5)', claude_text, elapsed))

# --- Gemini ---
start = time.time()
gemini_response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=question,
)
elapsed = time.time() - start
results.append(('Gemini (2.5-flash)', gemini_response.text, elapsed))

# --- Groq ---
start = time.time()
groq_response = groq_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role': 'user', 'content': question}],
)
elapsed = time.time() - start
results.append(('Groq (llama-3.1-8b-instant)', groq_response.choices[0].message.content, elapsed))

# --- Resultats ---
print(f'Question posee : {question}\n')
for name, text, elapsed in results:
    print(f'--- {name} ({elapsed:.2f}s) ---')
    print(text.strip())
    print()

## Comparaison sur une question plus complexe (raisonnement)

Un probleme de logique classique (poules et lapins) qui demande un vrai raisonnement etape par etape, pas juste un calcul direct. Plus revelateur des differences de qualite entre modeles.

In [ ]:
hard_question = (
    "Dans une ferme, il y a des poules et des lapins. "
    "Au total, on compte 35 tetes et 94 pattes. "
    "Combien y a-t-il de poules et combien y a-t-il de lapins ? "
    "Explique ton raisonnement etape par etape avant de donner la reponse finale."
)

hard_results = []

# --- Claude ---
start = time.time()
claude_response = anthropic_client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=1024,
    messages=[{'role': 'user', 'content': hard_question}],
)
elapsed = time.time() - start
claude_text = ''.join(b.text for b in claude_response.content if b.type == 'text')
hard_results.append(('Claude (haiku-4-5)', claude_text, elapsed))

# --- Gemini ---
start = time.time()
gemini_response = gemini_client.models.generate_content(
    model='gemini-2.5-flash',
    contents=hard_question,
)
elapsed = time.time() - start
hard_results.append(('Gemini (2.5-flash)', gemini_response.text, elapsed))

# --- Groq ---
start = time.time()
groq_response = groq_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role': 'user', 'content': hard_question}],
)
elapsed = time.time() - start
hard_results.append(('Groq (llama-3.1-8b-instant)', groq_response.choices[0].message.content, elapsed))

# --- Resultats ---
print(f'Question posee :\n{hard_question}\n')
print('(Reponse correcte attendue : 23 poules et 12 lapins)\n')
for name, text, elapsed in hard_results:
    print(f'=== {name} ({elapsed:.2f}s) ===')
    print(text.strip())
    print()